[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h07_sixth_decimal.ipynb)

# The sixth decimal place: measure it yourself

In 1894 Albert Michelson repeated a remark that the future truths of physics would be found "in the sixth place of
decimals". The film follows the measurements that kept proving it, from sundials to atomic clocks. Here you can
try them: turn time into an angle, drain a water clock, read a vernier, weigh Rayleigh's nitrogen and find the
argon, work out Michelson's fringe shift, and try the challenge from the video.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})
print("Ready. Run the next cell.")

## 1 · A sundial measures an angle

The Sun goes once around the sky in a day: 360° in 24 hours. Turn an angle into time, and see how long one of the
twelve daylight "hours" lasted in a city at any latitude, in midsummer and midwinter.

In [ ]:
def hours(latitude=41.9):
    print(f"360° ÷ 24 = {360 / 24:.0f}° an hour · 1° = {24 * 60 / 360:.0f} minutes · 1′ = {24 * 3600 / 21600:.0f} seconds")
    out = []
    for name, dec in (("midsummer", 23.44), ("midwinter", -23.44)):
        x = -math.tan(math.radians(latitude)) * math.tan(math.radians(dec))
        x = max(-1.0, min(1.0, x))
        day = 2 * math.degrees(math.acos(x)) / 15
        out.append((name, day))
        print(f"{name}: {day:.2f} h of daylight → one of twelve hours lasts {day * 60 / 12:.0f} minutes")
    fig, ax = plt.subplots(figsize=(8, 2.6))
    for k, (name, day) in enumerate(out):
        ax.barh(k, day, left=12 - day / 2, color=TEAL, alpha=0.5)
        for j in range(13): ax.plot([12 - day / 2 + day * j / 12] * 2, [k - 0.4, k + 0.4], color=CHALK, lw=1)
    ax.set_yticks([0, 1], [n for n, _ in out]); ax.set_xlim(0, 24); ax.set_xlabel("clock time (hours)")
    ax.set_title(f"twelve daylight hours at {latitude:.1f}°"); plt.show()

interact(hours, latitude=FloatSlider(value=41.9, min=0, max=60, step=0.5, description="latitude °"));

## 2 · The water clock's problem

Water leaves a small hole at a speed proportional to the square root of its depth (Torricelli, 1644), so a pot
empties fast at first and slowly at the end. Make the rim wider than the base and watch the level fall more evenly.
The Karnak clock is about 44 cm across at the top and 23 cm at the bottom: a ratio of about 1.9.

In [ ]:
def drain(rim_over_base=1.0):
    n = 2000
    xs = [k / n for k in range(n + 1)]
    A = lambda x: (1 + (rim_over_base - 1) * x) ** 2          # the pot's cross-section at height x (base = 1)
    T = [0.0]
    for k in range(1, n + 1):
        xm = (xs[k] + xs[k - 1]) / 2
        T.append(T[-1] + A(xm) / math.sqrt(xm) / n)           # time to fall from x to 0 ∝ ∫ A(x)/√x dx
    tot = T[-1]
    time_since_full = [1 - v / tot for v in T]
    half = time_since_full[n // 2]
    print(f"top half takes {half:.2f} of the time · bottom half {1 - half:.2f} · ratio {(1 - half) / half:.2f}")
    if rim_over_base == 1.0: print(f"straight sides: exactly 1 + √2 = {1 + math.sqrt(2):.3f}")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(time_since_full, xs, color=CORAL, lw=2.5, label=f"rim ÷ base = {rim_over_base:.2f}")
    ax.plot([0, 1], [1, 0], color=DIM, ls="--", label="perfectly even")
    ax.set_xlabel("time (fraction of the whole)"); ax.set_ylabel("water level"); ax.legend(); plt.show()

interact(drain, rim_over_base=FloatSlider(value=1.0, min=1.0, max=3.0, step=0.05, description="rim ÷ base"));

## 3 · How small is an arcminute?

A 2.5 cm coin covers a smaller angle the further away it is. Pick an angle in arcminutes (Tycho: about 1; Kepler's
gap: 8; Mercury's left-over turn a century: 43 arcseconds = 0.72′).

In [ ]:
def coin(arcminutes=1.0):
    d = 0.025 / math.tan(math.radians(arcminutes / 60))
    print(f"a 2.5 cm coin covers {arcminutes}′ from {d:,.1f} m away")
    print(f"as a fraction of a full circle: {arcminutes} ÷ 21,600 = {arcminutes / 21600:.7f}")

interact(coin, arcminutes=FloatSlider(value=1.0, min=0.1, max=30, step=0.1, description="arcminutes"));

## 4 · Read between the marks: the vernier

Ten sliding marks fit in the space of nine main marks, so each is short by a tenth. Slide it, and find the one pair
that lines up.

In [ ]:
def vernier(slide=0.7):
    fig, ax = plt.subplots(figsize=(9, 2.4))
    for k in range(13):
        ax.plot([k, k], [0, 0.6 if k % 5 else 0.9], color=CHALK, lw=2)
    best = min(range(11), key=lambda j: abs((slide + 0.9 * j) - round(slide + 0.9 * j)))
    for j in range(11):
        x = slide + 0.9 * j
        ax.plot([x, x], [-0.6, 0], color=CORAL if j == best else LILAC, lw=3 if j == best else 2)
    ax.set_ylim(-1, 1.2); ax.set_xlim(-0.5, 12.5); ax.set_yticks([]); ax.grid(False)
    ax.set_title(f"pair {best} lines up → the slide moved {best / 10:.1f} of a mark (true: {slide:.2f})"); plt.show()

interact(vernier, slide=FloatSlider(value=0.7, min=0, max=0.99, step=0.01, description="slide"));

## 5 · Mars is nearly a circle

Draw Mars's orbit a metre wide. How far does its short axis fall short of a circle, and how far is the Sun from the
middle? Mars: eccentricity 0.0934 (NASA). Try the Earth (0.0167) and Mercury (0.2056).

In [ ]:
def orbit(e=0.0934):
    a = 0.5
    b = a * math.sqrt(1 - e * e)
    print(f"short axis: {1000 * 2 * (a - b):.1f} mm less than 1 m · Sun {100 * a * e:.1f} cm from the middle")
    th = [2 * math.pi * k / 400 for k in range(401)]
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot([a * math.cos(t) for t in th], [a * math.sin(t) for t in th], color=DIM, label="circle")
    ax.plot([a * math.cos(t) for t in th], [b * math.sin(t) for t in th], color=SAGE, lw=2, label="the orbit")
    ax.plot([a * e], [0], "o", color=CORAL, ms=10, label="Sun"); ax.set_aspect("equal"); ax.legend(); plt.show()

interact(orbit, e=FloatSlider(value=0.0934, min=0, max=0.6, step=0.0001, description="eccentricity", readout_format=".4f"));

## 6 · Weigh Rayleigh's nitrogen and find the argon

Rayleigh's globe held 2.3102 g of "nitrogen" from air and 2.2990 g of nitrogen from compounds (1895 means). Argon
(39.948) is heavier than nitrogen gas (28.014). How much argon makes the air's nitrogen that much heavier?

In [ ]:
def argon(from_air=2.3102, from_compounds=2.2990):
    gap = (from_air - from_compounds) / from_compounds
    heavier = 39.948 / 28.014 - 1
    print(f"the air's nitrogen is heavier by {100 * gap:.2f} %")
    print(f"argon is heavier than nitrogen by {100 * heavier:.1f} %")
    print(f"so the hidden argon is about {100 * gap:.2f} % ÷ {100 * heavier:.1f} % = {100 * gap / heavier:.2f} % of it")
    print(f"today's measurement: {100 * 0.934 / (78.084 + 0.934):.2f} % (NASA, by volume)")

interact(argon, from_air=FloatSlider(value=2.3102, min=2.29, max=2.33, step=0.0001, readout_format=".4f"),
         from_compounds=FloatSlider(value=2.2990, min=2.28, max=2.32, step=0.0001, readout_format=".4f"));

## 7 · Michelson's expected shift, and LIGO's

Turning the 1887 instrument should have slid the stripes by 2·D·(v/c)² / λ of a stripe: D the light's path in each
arm, v the Earth's speed, λ the light's wavelength. Then LIGO: arms 4 km long, stretched by one part in 10²¹.

In [ ]:
def shift(path_m=11.0, speed_km_s=30.0, wavelength_nm=550.0):
    c = 299_792_458
    s = 2 * path_m * (speed_km_s * 1000 / c) ** 2 / (wavelength_nm * 1e-9)
    print(f"(v/c)² = {(speed_km_s * 1000 / c) ** 2:.2e}")
    print(f"expected shift: {s:.2f} of a stripe (they saw less than a fortieth of 0.4)")
    dl = 4000 * 1e-21
    print(f"LIGO: 4 km × 10⁻²¹ = {dl:.0e} m, about 1/{2 * 0.8408e-15 / dl:.0f} of a proton's width")

interact(shift, path_m=FloatSlider(value=11.0, min=1, max=30, step=0.5, description="D (m)"),
         speed_km_s=FloatSlider(value=30.0, min=1, max=60, step=1, description="v (km/s)"),
         wavelength_nm=FloatSlider(value=550, min=400, max=700, step=10, description="λ (nm)"));

## 8 · The challenge from the video

A vernier scale has twenty sliding marks that fit in the space of nineteen main marks. What's the smallest step it
can read? Think first, then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("20 sliding marks in the space of 19: the smallest step?   (press the button when you're ready)"); return
    print(f"each sliding mark is 19/20 of a main mark, short by {1 - 19 / 20:.2f}")
    print("so the mismatch grows by 1/20 at each step: on a millimetre scale, 0.05 mm")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
A. A. Michelson, *Quarterly Calendar* (U. Chicago), Aug 1894; Michelson & Morley, *Am. J. Sci.* 34 (1887). Rayleigh,
*Proc. Roy. Inst.* 14 (1895). NASA planetary and Earth fact sheets. LIGO, *Phys. Rev. Lett.* 116, 061102 (2016);
CODATA proton radius. McMaster Ancient Egyptian Astronomy (the Karnak clock). All retrieved 2026-10-30/31. Every
number above is computed in the cells.